# Project 6 — Extended Solution: Production-minded Widget errors
**Python 3.11+ | Standard library only | Fully standalone**

The source exercise asks for an exception hierarchy, customer/internal messages, HTTP status codes, JSON responses, logging, and tracebacks. This version implements all of that and then adds a framework-neutral HTTP response adapter, stable application error codes, request IDs, redacted diagnostic context, retriable-delays metadata, a machine-readable exception catalog, and an atomic concurrent checkout demo.

## Architecture and threat model
- **Required taxonomy and HTTP mapping:** all errors remain HTTP 500 except invalid/stacked coupons (400). Real APIs may choose 409 or 503 in some situations; the exercise's explicit mapping takes precedence.
- **Privacy by construction:** `to_dict()` uses a strict allowlist of public fields. Internal `message`, diagnostic `context`, traceback, and source lines are **never** included. A custom customer message must be intentionally supplied by trusted application code.
- **Operations:** each exception gets a validated UUID request identifier and stable machine error code; customers can quote the request ID for support.
- **Logging:** standard `logging`, UTC timestamps, bounded recursive context sanitation, common-secret-key redaction, `TracebackException(..., capture_locals=False)`, and chained traceback reporting. Internal message and source lines are **not fully redactable**, so logs require access control and a retention policy. Never put credentials into exception messages.
- **Retry metadata:** supplier delay errors can advertise `retryable` and a positive `Retry-After` header. This does **not** silently change their required HTTP 500 status.
- **Concurrency:** the example checkout service uses `RLock` to serialize stock check + decrement; this is an in-process illustration, **not** a substitute for database transactions or distributed locks.
- **Limitations:** the sample sanitizer recognizes common secret keys, shallow-copies the context, and is not a full DLP system. HTTP response objects are immutable shells, while strings are inherently immutable.

## Fully independent exception library

In [1]:
from __future__ import annotations

from collections.abc import Mapping
from dataclasses import dataclass
from datetime import datetime, timezone
from http import HTTPStatus
import json
import logging
from threading import RLock
import traceback as traceback_module
from types import MappingProxyType
from typing import ClassVar
from uuid import UUID, uuid4


_SENSITIVE_KEYS = frozenset({
    "password", "passwd", "secret", "token", "api_key", "authorization",
    "cookie", "card_number", "credit_card", "cvv", "access_token",
})


def _safe_log_value(value: object, *, depth: int = 0) -> object:
    """Create bounded JSON-safe diagnostics while redacting common sensitive keys.

    This is defense in depth, NOT a substitute for an organization's log-policy.
    No arbitrary object is repr()-serialized; unsupported objects expose only type.
    """
    if depth >= 5:
        return "<max-depth>"
    if isinstance(value, Mapping):
        result: dict[str, object] = {}
        for key, child in list(value.items())[:40]:
            name = str(key)
            result[name] = (
                "[REDACTED]" if name.lower().replace("-", "_") in _SENSITIVE_KEYS
                else _safe_log_value(child, depth=depth + 1)
            )
        return result
    if isinstance(value, (list, tuple)):
        return [_safe_log_value(child, depth=depth + 1) for child in value[:40]]
    if value is None or isinstance(value, (bool, int, float, str)):
        return value[:300] + "…" if isinstance(value, str) and len(value) > 300 else value
    return f"<{type(value).__name__}>"


@dataclass(frozen=True, slots=True)
class HTTPResponse:
    """Framework-neutral HTTP response representation (no live server required)."""

    status: int
    body: str
    headers: Mapping[str, str]

    def __post_init__(self) -> None:
        object.__setattr__(self, "headers", MappingProxyType(dict(self.headers)))


class WidgetException(Exception):
    """Secure, traceable and reusable base class for widget errors."""

    default_message: ClassVar[str] = "Unexpected widget application error."
    default_customer_message: ClassVar[str] = "We could not complete your request."
    http_status: ClassVar[HTTPStatus] = HTTPStatus.INTERNAL_SERVER_ERROR
    error_code: ClassVar[str] = "WID-000"
    retryable: ClassVar[bool] = False

    def __init__(
        self,
        message: str | None = None,
        *,
        customer_message: str | None = None,
        context: Mapping[str, object] | None = None,
        request_id: str | UUID | None = None,
        occurred_at: datetime | None = None,
        retry_after: int | None = None,
    ) -> None:
        self.message = self._message(message, "message", self.default_message)
        self.customer_message = self._message(
            customer_message, "customer_message", self.default_customer_message
        )
        if context is not None and not isinstance(context, Mapping):
            raise TypeError("context must be a mapping or None")
        copied = dict(context or {})
        if not all(isinstance(key, str) for key in copied):
            raise TypeError("context keys must be strings")
        self.context = MappingProxyType(copied)
        if request_id is None:
            self.request_id = str(uuid4())
        else:
            try:
                self.request_id = str(UUID(str(request_id)))
            except (TypeError, ValueError, AttributeError) as exc:
                raise ValueError("request_id must be a valid UUID") from exc
        if occurred_at is None:
            occurred_at = datetime.now(timezone.utc)
        if not isinstance(occurred_at, datetime) or occurred_at.tzinfo is None or occurred_at.utcoffset() is None:
            raise ValueError("occurred_at must be a timezone-aware datetime")
        self.occurred_at = occurred_at.astimezone(timezone.utc)
        if retry_after is not None:
            if type(retry_after) is not int or retry_after <= 0:
                raise ValueError("retry_after must be a positive integer in seconds")
            if not self.retryable:
                raise ValueError("retry_after is only valid on retryable errors")
        self.retry_after = retry_after
        super().__init__(self.message)

    @staticmethod
    def _message(value: str | None, name: str, fallback: str) -> str:
        if value is None:
            return fallback
        if not isinstance(value, str):
            raise TypeError(f"{name} must be a string or None")
        if not value.strip():
            raise ValueError(f"{name} must not be blank")
        return value

    @property
    def traceback(self):
        """Lazy formatted traceback without captured local-variable values."""
        return traceback_module.TracebackException.from_exception(
            self, capture_locals=False
        ).format()

    def format_traceback(self) -> str:
        return "".join(self.traceback)

    def log_exception(self, logger: logging.Logger | None = None) -> dict[str, object]:
        """Record internal diagnostics, with bounded/redacted structured context."""
        target = logger if logger is not None else logging.getLogger("widget.exceptions")
        record: dict[str, object] = {
            "event": "widget_exception",
            "logged_at_utc": datetime.now(timezone.utc).isoformat(),
            "occurred_at_utc": self.occurred_at.isoformat(),
            "request_id": self.request_id,
            "error_code": self.error_code,
            "exception_type": type(self).__name__,
            "http_status": int(self.http_status),
            "retryable": self.retryable,
            "internal_message": self.message,
            "context": _safe_log_value(self.context),
            "traceback": self.format_traceback(),
        }
        # Internal message/traceback may contain secrets: send only to protected logs.
        target.error(json.dumps(record, ensure_ascii=False, allow_nan=False))
        return record

    def to_dict(self) -> dict[str, object]:
        """Explicit public allowlist. No diagnostics, context, or traceback."""
        return {
            "status": int(self.http_status),
            "error": {
                "code": self.error_code,
                "message": self.customer_message,
                "category": type(self).__name__,
                "request_id": self.request_id,
                "retryable": self.retryable,
            },
            "time_utc": self.occurred_at.isoformat(),
        }

    def to_json(self) -> str:
        return json.dumps(self.to_dict(), ensure_ascii=False)

    def to_http_response(self) -> HTTPResponse:
        headers = {
            "Content-Type": "application/json; charset=utf-8",
            "X-Request-ID": self.request_id,
            "Cache-Control": "no-store",
        }
        if self.retry_after is not None:
            headers["Retry-After"] = str(self.retry_after)
        return HTTPResponse(status=int(self.http_status), body=self.to_json(), headers=headers)


class SupplierException(WidgetException):
    default_message = "Supplier exception."
    error_code = "WID-100"


class NotManufacturedException(SupplierException):
    default_message = "Widget no longer manufactured."
    error_code = "WID-101"


class ProductionDelayedException(SupplierException):
    default_message = "Production delayed."
    error_code = "WID-102"
    retryable = True


class ShippingDelayedException(SupplierException):
    default_message = "Shipping delayed."
    error_code = "WID-103"
    retryable = True


class CheckoutException(WidgetException):
    default_message = "Checkout error."
    error_code = "WID-200"


class InventoryException(CheckoutException):
    default_message = "Checkout inventory error."
    error_code = "WID-210"


class OutOfStockException(InventoryException):
    default_message = "Inventory out of stock."
    error_code = "WID-211"


class PricingException(CheckoutException):
    default_message = "Checkout pricing error."
    error_code = "WID-220"


class InvalidCouponCodeException(PricingException):
    default_message = "Invalid coupon code."
    default_customer_message = "The coupon code is invalid."
    http_status = HTTPStatus.BAD_REQUEST
    error_code = "WID-221"


class CannotStackCouponException(PricingException):
    default_message = "Coupons cannot be stacked."
    default_customer_message = "Multiple coupon codes cannot be combined."
    http_status = HTTPStatus.BAD_REQUEST
    error_code = "WID-222"


def exception_catalog() -> tuple[dict[str, object], ...]:
    """Introspect the declared taxonomy and return a stable, sorted catalog."""
    def descendants(parent: type[WidgetException]):
        for subclass in parent.__subclasses__():
            yield subclass
            yield from descendants(subclass)

    classes = (WidgetException, *descendants(WidgetException))
    codes = [cls.error_code for cls in classes]
    if len(codes) != len(set(codes)):
        raise RuntimeError("Duplicate application error codes")
    return tuple(
        {
            "category": cls.__name__,
            "error_code": cls.error_code,
            "http_status": int(cls.http_status),
            "retryable": cls.retryable,
            "default_customer_message": cls.default_customer_message,
        }
        for cls in sorted(classes, key=lambda cls: cls.error_code)
    )

## Integration example — atomic checkout and framework-neutral response

In [2]:
@dataclass(frozen=True, slots=True)
class Receipt:
    sku: str
    quantity: int
    remaining_stock: int
    applied_coupon: str | None
    request_id: str


class CheckoutService:
    """In-memory demo: lock stock checks + inventory decrement atomically."""

    def __init__(self, stock: Mapping[str, int], valid_coupons: frozenset[str]):
        if not isinstance(stock, Mapping):
            raise TypeError("stock must be a mapping")
        if not isinstance(valid_coupons, frozenset) or not all(type(x) is str for x in valid_coupons):
            raise TypeError("valid_coupons must be a frozenset of strings")
        if not all(type(sku) is str and sku.strip() and type(count) is int and count >= 0
                   for sku, count in stock.items()):
            raise ValueError("stock must map nonempty SKU strings to nonnegative integer quantities")
        self._stock = dict(stock)
        self._coupons = valid_coupons
        self._lock = RLock()

    def get_stock(self, sku: str) -> int:
        with self._lock:
            return self._stock.get(sku, 0)

    def place_order(
        self, sku: str, quantity: int, *, coupons: tuple[str, ...] = (),
        request_id: str | UUID | None = None,
    ) -> Receipt:
        """Validate before committing. Failures leave stock unchanged."""
        if type(sku) is not str or not sku.strip():
            raise ValueError("sku must be a nonempty string")
        if type(quantity) is not int or quantity < 1:
            raise ValueError("quantity must be a positive integer")
        if not isinstance(coupons, tuple) or not all(type(code) is str for code in coupons):
            raise TypeError("coupons must be a tuple of strings")
        # Use one correlation ID throughout the request, including on success.
        try:
            correlation = str(uuid4() if request_id is None else UUID(str(request_id)))
        except (TypeError, ValueError, AttributeError) as exc:
            raise ValueError("request_id must be a valid UUID") from exc
        if len(coupons) > 1:
            raise CannotStackCouponException(
                context={"sku": sku, "coupon_count": len(coupons)},
                request_id=correlation,
            )
        if coupons and coupons[0] not in self._coupons:
            raise InvalidCouponCodeException(
                context={"sku": sku, "coupon": coupons[0]},
                request_id=correlation,
            )
        with self._lock:
            available = self._stock.get(sku, 0)
            if available < quantity:
                raise OutOfStockException(
                    context={"sku": sku, "requested": quantity, "available": available},
                    request_id=correlation,
                )
            self._stock[sku] = available - quantity
            return Receipt(sku, quantity, self._stock[sku], coupons[0] if coupons else None, correlation)


def handle_checkout(
    service: CheckoutService,
    sku: str,
    quantity: int,
    *,
    coupons: tuple[str, ...] = (),
    request_id: str | UUID | None = None,
    logger: logging.Logger | None = None,
) -> HTTPResponse:
    """Tiny framework adapter: actual status and body agree; log once on failure."""
    try:
        receipt = service.place_order(sku, quantity, coupons=coupons, request_id=request_id)
    except WidgetException as exc:
        exc.log_exception(logger)
        return exc.to_http_response()
    return HTTPResponse(
        status=int(HTTPStatus.OK),
        body=json.dumps({
            "sku": receipt.sku,
            "quantity": receipt.quantity,
            "remaining_stock": receipt.remaining_stock,
            "applied_coupon": receipt.applied_coupon,
            "request_id": receipt.request_id,
        }),
        headers={
            "Content-Type": "application/json; charset=utf-8",
            "X-Request-ID": receipt.request_id,
            "Cache-Control": "no-store",
        },
    )

In [3]:
# Production frameworks should populate request_id from trusted request middleware.
service = CheckoutService({"RYZEN-CPU": 2}, frozenset({"SAVE10"}))
log_sink = logging.getLogger("widget.example")

success = handle_checkout(service, "RYZEN-CPU", 1, coupons=("SAVE10",), logger=log_sink)
assert success.status == 200
print("Successful order:", success.body)

# Catch the expected failure without emitting a demo traceback into notebook output.
from unittest.mock import Mock
error_logger = Mock(spec=logging.Logger)
failure = handle_checkout(service, "RYZEN-CPU", 1, coupons=("BAD",), logger=error_logger)
assert failure.status == 400
assert "context" not in failure.body
print("Invalid coupon response:", failure.body)

print("Available stock after success + rejected checkout:", service.get_stock("RYZEN-CPU"))
print("Catalog entries:", len(exception_catalog()))

Successful order: {"sku": "RYZEN-CPU", "quantity": 1, "remaining_stock": 1, "applied_coupon": "SAVE10", "request_id": "d3ca1e02-80c9-4906-be46-dc1a380486cc"}
Invalid coupon response: {"status": 400, "error": {"code": "WID-221", "message": "The coupon code is invalid.", "category": "InvalidCouponCodeException", "request_id": "bf519478-d22b-4d2e-b814-de8f2460fa7c", "retryable": false}, "time_utc": "2026-11-05T17:07:39.180720+00:00"}
Available stock after success + rejected checkout: 1
Catalog entries: 11


## Additional test suite
The tests validate correct inheritance, the exact assignment-defined HTTP mapping, response/log separation, stable codes, UUIDs, timestamps, sanitization, retries, traceback and chaining, no-stock-change failures, the HTTP adapter, and a contention test showing inventory cannot be oversold in one process.

In [4]:
from concurrent.futures import ThreadPoolExecutor
import unittest
from unittest.mock import Mock


class TestExtendedWidget(unittest.TestCase):
    def test_hierarchy(self):
        self.assertTrue(issubclass(OutOfStockException, CheckoutException))
        self.assertTrue(issubclass(OutOfStockException, InventoryException))
        self.assertTrue(issubclass(InvalidCouponCodeException, PricingException))
        self.assertTrue(issubclass(ProductionDelayedException, SupplierException))

    def test_status_contract(self):
        for row in exception_catalog():
            expected = 400 if row["category"] in {"InvalidCouponCodeException", "CannotStackCouponException"} else 500
            with self.subTest(row=row):
                self.assertEqual(row["http_status"], expected)

    def test_code_catalog_is_unique_sorted(self):
        catalog = exception_catalog()
        codes = [entry["error_code"] for entry in catalog]
        self.assertEqual(len(codes), 11)
        self.assertEqual(len(set(codes)), 11)
        self.assertEqual(codes, sorted(codes))

    def test_internal_message_not_public(self):
        exc = WidgetException("SUPER_SECRET_MESSAGE", context={"token": "SUPER_SECRET_TOKEN"})
        self.assertNotIn("SUPER_SECRET", exc.to_json())
        self.assertNotIn("traceback", exc.to_json())
        self.assertEqual(exc.args, ("SUPER_SECRET_MESSAGE",))

    def test_public_override(self):
        exc = InvalidCouponCodeException("diagnostic", customer_message="Coupon expired")
        self.assertEqual(exc.to_dict()["error"]["message"], "Coupon expired")
        self.assertNotIn("diagnostic", exc.to_json())

    def test_blank_or_bad_messages(self):
        with self.assertRaises(ValueError):
            WidgetException(" ")
        with self.assertRaises(TypeError):
            WidgetException(customer_message=10)
        with self.assertRaises(ValueError):
            WidgetException(customer_message="")

    def test_mapping_is_defensively_copied(self):
        external = {"sku": "A"}
        exc = WidgetException(context=external)
        external["sku"] = "B"
        self.assertEqual(exc.context["sku"], "A")
        with self.assertRaises(TypeError):
            exc.context["sku"] = "C"

    def test_mapping_invalid(self):
        with self.assertRaises(TypeError):
            WidgetException(context=[1])
        with self.assertRaises(TypeError):
            WidgetException(context={1: "bad"})

    def test_request_id_generated_valid_uuid(self):
        exc = WidgetException()
        self.assertEqual(str(UUID(exc.request_id)), exc.request_id)

    def test_request_id_canonicalized(self):
        request = "A0B1C2D3-E4F5-4A67-8B90-0123456789AB"
        exc = WidgetException(request_id=request)
        self.assertEqual(exc.request_id, request.lower())
        self.assertEqual(exc.to_dict()["error"]["request_id"], request.lower())

    def test_invalid_request_id(self):
        with self.assertRaises(ValueError):
            WidgetException(request_id="invalid")

    def test_utc_time_converts_and_stays_stable(self):
        from datetime import timedelta
        local = datetime(2026, 1, 1, 12, tzinfo=timezone(timedelta(hours=2)))
        exc = WidgetException(occurred_at=local)
        self.assertEqual(exc.to_dict()["time_utc"], "2026-01-01T10:00:00+00:00")
        self.assertEqual(exc.to_dict(), json.loads(exc.to_json()))

    def test_rejects_naive_datetime(self):
        with self.assertRaises(ValueError):
            WidgetException(occurred_at=datetime(2026, 1, 1))

    def test_retryability_and_header(self):
        delayed = ProductionDelayedException(retry_after=120)
        response = delayed.to_http_response()
        self.assertEqual(response.status, 500)
        self.assertEqual(response.headers["Retry-After"], "120")
        self.assertTrue(json.loads(response.body)["error"]["retryable"])

    def test_nonretryable_cannot_set_retry_after(self):
        with self.assertRaises(ValueError):
            InvalidCouponCodeException(retry_after=10)
        for invalid in (0, -1, True, "3"):
            with self.subTest(invalid=invalid), self.assertRaises(ValueError):
                ShippingDelayedException(retry_after=invalid)

    def test_response_headers_are_read_only_copy(self):
        headers = {"X-Test": "value"}
        response = HTTPResponse(200, "{}", headers)
        headers["X-Test"] = "changed"
        self.assertEqual(response.headers["X-Test"], "value")
        with self.assertRaises(TypeError):
            response.headers["X-Test"] = "x"

    def test_response_status_matches_body(self):
        error = InvalidCouponCodeException()
        response = error.to_http_response()
        self.assertEqual(response.status, json.loads(response.body)["status"])
        self.assertEqual(response.headers["X-Request-ID"], error.request_id)
        self.assertEqual(response.headers["Cache-Control"], "no-store")

    def test_logging_redacts_secrets_nested(self):
        logger = Mock(spec=logging.Logger)
        error = WidgetException(
            "private diagnostic", context={"api_key": "NOPE", "nested": {"token": "HIDDEN", "sku": "AB"}}
        )
        error.log_exception(logger)
        record = json.loads(logger.error.call_args.args[0])
        self.assertEqual(record["context"]["api_key"], "[REDACTED]")
        self.assertEqual(record["context"]["nested"]["token"], "[REDACTED]")
        self.assertEqual(record["context"]["nested"]["sku"], "AB")
        self.assertNotIn("HIDDEN", json.dumps(record))

    def test_log_unsupported_objects_not_repr(self):
        class SecretObject:
            def __repr__(self):
                return "DO_NOT_PRINT_SECRET"
        logger = Mock(spec=logging.Logger)
        WidgetException(context={"obj": SecretObject()}).log_exception(logger)
        output = logger.error.call_args.args[0]
        self.assertNotIn("DO_NOT_PRINT_SECRET", output)
        self.assertIn("<SecretObject>", output)

    def test_sanitizer_bounds(self):
        self.assertEqual(len(_safe_log_value("a" * 500)), 301)
        self.assertEqual(_safe_log_value([0] * 100).__len__(), 40)
        self.assertEqual(_safe_log_value({"a": {"b": {"c": {"d": {"e": "z"}}}}})["a"]["b"]["c"]["d"]["e"], "<max-depth>")

    def test_traceback_raised_and_logged(self):
        logger = Mock(spec=logging.Logger)
        try:
            raise OutOfStockException("diagnostic")
        except OutOfStockException as exc:
            record = exc.log_exception(logger)
            text = exc.format_traceback()
            public_json = exc.to_json()
        self.assertIn("Traceback (most recent call last)", text)
        self.assertIn("OutOfStockException", record["traceback"])
        self.assertNotIn("traceback", public_json)

    def test_traceback_chaining(self):
        try:
            try:
                raise ValueError("root cause")
            except ValueError as cause:
                raise ProductionDelayedException("downstream") from cause
        except ProductionDelayedException as exc:
            text = exc.format_traceback()
        self.assertIn("direct cause", text)
        self.assertIn("ValueError", text)

    def test_unraised_has_no_traceback_frames(self):
        self.assertNotIn("Traceback (most recent call last)", WidgetException().format_traceback())

    def test_stock_constructor_validates(self):
        for stock in ({"A": -1}, {"A": True}, {"": 1}):
            with self.subTest(stock=stock), self.assertRaises(ValueError):
                CheckoutService(stock, frozenset())
        with self.assertRaises(TypeError):
            CheckoutService({"A": 1}, {"SAVE"})

    def test_simple_checkout_reduces_stock(self):
        service = CheckoutService({"CPU": 3}, frozenset({"SAVE10"}))
        receipt = service.place_order("CPU", 2, coupons=("SAVE10",))
        self.assertEqual(receipt.remaining_stock, 1)
        self.assertEqual(receipt.applied_coupon, "SAVE10")
        self.assertEqual(service.get_stock("CPU"), 1)

    def test_cannot_stack_does_not_modify_stock(self):
        service = CheckoutService({"CPU": 3}, frozenset({"A", "B"}))
        with self.assertRaises(CannotStackCouponException) as captured:
            service.place_order("CPU", 1, coupons=("A", "B"))
        self.assertEqual(captured.exception.http_status, 400)
        self.assertEqual(service.get_stock("CPU"), 3)

    def test_invalid_coupon_does_not_modify_stock(self):
        service = CheckoutService({"CPU": 3}, frozenset({"A"}))
        with self.assertRaises(InvalidCouponCodeException):
            service.place_order("CPU", 1, coupons=("NOPE",))
        self.assertEqual(service.get_stock("CPU"), 3)

    def test_out_of_stock_does_not_modify_stock(self):
        service = CheckoutService({"CPU": 3}, frozenset())
        with self.assertRaises(OutOfStockException) as captured:
            service.place_order("CPU", 4)
        self.assertEqual(service.get_stock("CPU"), 3)
        self.assertEqual(captured.exception.context["requested"], 4)

    def test_invalid_checkout_inputs_do_not_mutate_stock(self):
        service = CheckoutService({"CPU": 3}, frozenset())
        for quantity in (0, -1, True, 1.5):
            with self.subTest(quantity=quantity), self.assertRaises(ValueError):
                service.place_order("CPU", quantity)
        with self.assertRaises(TypeError):
            service.place_order("CPU", 1, coupons=["A"])
        self.assertEqual(service.get_stock("CPU"), 3)

    def test_handler_error_maps_http_status_and_correlation(self):
        logger = Mock(spec=logging.Logger)
        service = CheckoutService({"CPU": 2}, frozenset({"SAVE10"}))
        rid = str(uuid4())
        response = handle_checkout(service, "CPU", 1, coupons=("BAD",), request_id=rid, logger=logger)
        self.assertEqual(response.status, 400)
        self.assertEqual(response.headers["X-Request-ID"], rid)
        self.assertEqual(json.loads(response.body)["error"]["request_id"], rid)
        self.assertEqual(logger.error.call_count, 1)
        self.assertEqual(service.get_stock("CPU"), 2)

    def test_handler_success_does_not_log(self):
        logger = Mock(spec=logging.Logger)
        service = CheckoutService({"CPU": 2}, frozenset())
        response = handle_checkout(service, "CPU", 1, logger=logger)
        self.assertEqual(response.status, 200)
        self.assertEqual(json.loads(response.body)["remaining_stock"], 1)
        logger.error.assert_not_called()

    def test_concurrent_orders_never_oversell(self):
        service = CheckoutService({"CPU": 10}, frozenset())
        def attempt(_):
            try:
                service.place_order("CPU", 1)
                return True
            except OutOfStockException:
                return False
        with ThreadPoolExecutor(max_workers=12) as pool:
            successes = list(pool.map(attempt, range(30)))
        self.assertEqual(sum(successes), 10)
        self.assertEqual(service.get_stock("CPU"), 0)


result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestExtendedWidget)
)
assert result.wasSuccessful(), "Extended suite failed"
print(f"Extended solution: {result.testsRun} tests passed")

test_blank_or_bad_messages (__main__.TestExtendedWidget.test_blank_or_bad_messages) ... ok
test_cannot_stack_does_not_modify_stock (__main__.TestExtendedWidget.test_cannot_stack_does_not_modify_stock) ... ok
test_code_catalog_is_unique_sorted (__main__.TestExtendedWidget.test_code_catalog_is_unique_sorted) ... ok
test_concurrent_orders_never_oversell (__main__.TestExtendedWidget.test_concurrent_orders_never_oversell) ... ok
test_handler_error_maps_http_status_and_correlation (__main__.TestExtendedWidget.test_handler_error_maps_http_status_and_correlation) ... ok
test_handler_success_does_not_log (__main__.TestExtendedWidget.test_handler_success_does_not_log) ... ok
test_hierarchy (__main__.TestExtendedWidget.test_hierarchy) ... ok
test_internal_message_not_public (__main__.TestExtendedWidget.test_internal_message_not_public) ... ok
test_invalid_checkout_inputs_do_not_mutate_stock (__main__.TestExtendedWidget.test_invalid_checkout_inputs_do_not_mutate_stock) ... ok
test_invalid_coupon_d

Extended solution: 32 tests passed


## Adapting to FastAPI, Flask, or another framework
1. Generate a trusted request ID in middleware and pass it to domain operations.
2. Register a handler for `WidgetException`. In it, call `exc.log_exception(protected_logger)` **once** and `response = exc.to_http_response()`; set the real HTTP status to `response.status` and copy its headers and body into your framework's response.
3. Catch uncaught unexpected errors in a **separate** top-level handler and return a generic 500. Do not catch `BaseException` or swallow process-control exceptions.
4. Replace in-memory inventory with a transactional database for multi-process deployments; provide audited persistent event IDs, rate limits, metrics, and purpose-built redaction.
5. When using this with a live API, review security, observability, and contractual status-code policies; do not expose internal exceptions or tracebacks.

**Important:** This is an educational, self-contained notebook; it makes no network calls and does not install packages. Run top to bottom after a kernel restart.